# MetabTravLR quickstart

Train SpaceTravLR with **harreman metabolite transporter pairs** added as a modulator
group, then read the learned `beta_<export>@<import>` coefficients back out over labeled
gene sets to rank metabolites by effect. We analyze the coefficients directly — **no
perturbation**.

Edit the **Config** and **Gene sets** cells, then run top to bottom. Everything is written
under the dataset directory. On Savio, replace `fit(...)` with the `spawn_worker` cell.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

# SpaceTravLR package (src/) + our metab_processing helpers
_here = os.path.dirname(os.path.abspath('.'))
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
sys.path.append(os.path.join(os.getcwd(), '..'))

from SpaceTravLR.spaceship import SpaceShip
from metab_processing.metab_loader import load_metab_pairs
from metab_processing import beta_analysis

## Config — data dir / dataset selection

Layout assumed: `DATA_DIR / DATASET / {adata.h5ad, easy_download/harreman_outputs/...}`.
Results are written to `DATA_DIR / DATASET / spacetravlr_output`.

In [ ]:
DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data'
DATASET  = 'Xenium/Primary_Dermal_Melanoma'   # dataset folder under DATA_DIR

CELL_TYPE_SRC = 'leiden_scVI_res_0.5'   # adata.obs column to use as 'cell_type' (the harreman tier annotation)

dataset_dir    = f'{DATA_DIR}/{DATASET}'
adata_path     = f'{dataset_dir}/adata.h5ad'
harreman_dir   = f'{dataset_dir}/easy_download/harreman_outputs'
selection_yaml = f'{harreman_dir}/metabolite_selection.yaml'
outdir         = f'{dataset_dir}/spacetravlr_output'
betadata_dir   = f'{outdir}/betadata'

for p in (adata_path, selection_yaml):
    assert os.path.exists(p), f'missing: {p}'

## Gene sets

The target genes to train and the labels to score metabolites against. `focus_genes` (the
genes actually trained) is the union of all sets. **Edit these lists.** With exactly
`positive`/`negative` labels, the ranking uses `signed = positive − negative`.

In [ ]:
GENE_SETS = {
    'positive': ['CD4', 'CD3E', 'IL2RA'],          # e.g. T-cell activity
    'negative': ['CTLA4', 'FOXP3', 'IL10', 'ENTPD1'],  # e.g. exhaustion
}

focus_genes = list(dict.fromkeys(g for genes in GENE_SETS.values() for g in genes))
print(f'{len(focus_genes)} focus genes:', focus_genes)

In [ ]:
adata = sc.read_h5ad(adata_path)
adata.obs['cell_type'] = adata.obs[CELL_TYPE_SRC]
adata.layers['raw_count'] = adata.X
adata

## Metabolite pairs from harreman

`metabolite_selection.yaml` → the deduped `metab_pairs` list (homotypic once, heterotypic
both orientations) filtered to genes in the panel. `selection` keeps the metabolite→pairs
grouping for the read-back.

In [ ]:
metab_pairs, selection = load_metab_pairs(selection_yaml, var_names=adata.var_names)
print(f'{len(selection)} metabolites, {len(metab_pairs)} model pairs (both orientations, in-panel)')
metab_pairs[:8]

## Setup + train

COMMOT is skipped — harreman is our metabolite prior. Only `focus_genes` are trained.

In [ ]:
spacetravlr = SpaceShip(
    name=DATASET.replace('/', '_'),
    outdir=outdir,
    genes=focus_genes,
)

In [ ]:
spacetravlr.setup_(adata, overwrite=False, run_commot=False)
assert spacetravlr.is_everything_ok()

In [ ]:
# Local / single-process training. On Savio use the spawn_worker cell below instead.
spacetravlr.fit(metab_pairs=metab_pairs)

In [ ]:
# --- Savio: run this cell (multiple times) to spawn parallel SLURM workers instead of fit() ---
# spacetravlr.focus_genes = focus_genes
# spacetravlr.spawn_worker(
#     account='fc_wagnerlabfca',
#     partition='savio4_gpu',
#     qos='a5k_gpu4_normal',
#     gres='gpu:A5000:1',
#     job_name='MetabTravLR',
#     cpus_per_task=4,
#     lifespan=0.5,
#     python_path='/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python',
#     metab_pairs=metab_pairs,   # if driving via a launch.py, pass metab_pairs to run_spacetravlr
# )

## Read the metabolite coefficients back out

Per-`(gene, pair, cell_type)` beta summary → roll up to metabolites (optionally weighting
each transporter pair by its harreman `C_np` communication score) → signed gene-set ranking.

In [ ]:
# Per-(gene, export, import, cell_type) mean/std of the learned metabolite beta.
pair_summary = beta_analysis.read_metab_beta_summary(
    betadata_dir,
    genes=focus_genes,
    obs=adata.obs,
    cell_type_col='cell_type',
)
pair_summary.head()

In [ ]:
# Optional: weight transporter pairs by harreman C_np (real-communication score).
# Set weights=None for a plain mean across a metabolite's pairs.
try:
    weights = beta_analysis.gene_pair_cnp_weights(harreman_dir, agg='max')
except Exception as e:
    print(f'C_np weights unavailable ({e}); falling back to unweighted mean')
    weights = None

metab_summary = beta_analysis.aggregate_to_metabolite(pair_summary, selection, weights=weights)
metab_summary.head()

In [ ]:
# Signed metabolite ranking: mean over 'positive' genes − mean over 'negative' genes.
ranking = beta_analysis.gene_set_score(metab_summary, GENE_SETS)
ranking.head(20)